In [1]:
import os
import argparse
import random
import time
import warnings
warnings.filterwarnings("ignore")
 
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms.functional import rotate as tvrotate
from torchvision import transforms
from tqdm.auto import tqdm

def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
 
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def get_args():
    p = argparse.ArgumentParser(description="Bonus Task – Exp 4: Algebra-Commutant Equivariant Classifier")
    p.add_argument("--task1_ckpt",      type=str, default="/kaggle/input/models/vitthal2945/e2e-vaemodel/pytorch/default/1/best_model.pt")
    p.add_argument("--task2_ckpt",      type=str, default="/kaggle/input/models/vitthal2945/best-model-task2/pytorch/default/1/best_model_task2.pt")
    p.add_argument("--data_dir",        type=str, default="/kaggle/input/datasets/vitthal2945/e2e-hidden-symmetries-dataset")
    p.add_argument("--out_dir",         type=str, default="./bonus_outputs/exp4")
    p.add_argument("--latent_dim",      type=int, default=16)
    p.add_argument("--n_classes",       type=int, default=2)
    p.add_argument("--epochs",          type=int, default=40)
    p.add_argument("--batch_size",      type=int, default=256)
    p.add_argument("--lr",              type=float, default=5e-4)
    p.add_argument("--n_reynolds",      type=int, default=12,
                   help="Number of discrete group elements for Reynolds operator (Z_n)")
    p.add_argument("--lambda_commutator", type=float, default=0.1,
                   help="Weight of soft commutator regularisation loss")
    p.add_argument("--projection_mode", type=str, default="both",
                   choices=["hard", "soft", "both", "none"],
                   help="hard=project weights; soft=add commutator loss; both=combined")
    return p.parse_args([])

In [3]:
class ResBlock(nn.Module):
    def __init__(self, ch, g=8):
        super().__init__()
        g = min(g, ch)
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch),
        )
        self.act = nn.SiLU()
    def forward(self, x): return self.act(x + self.net(x))
 
class Encoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.stem  = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.SiLU())
        self.l1    = nn.Sequential(ResBlock(32),  nn.Conv2d(32,  64, 4, 2, 1, bias=False), nn.GroupNorm(8,  64), nn.SiLU())
        self.l2    = nn.Sequential(ResBlock(64),  nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.SiLU())
        self.l3    = ResBlock(128)
        self.fc    = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 512), nn.SiLU())
        self.fc_mu = nn.Linear(512, ld); self.fc_lv = nn.Linear(512, ld)
    def forward(self, x):
        h = self.l3(self.l2(self.l1(self.stem(x)))); h = self.fc(h)
        return self.fc_mu(h), self.fc_lv(h)
 
class Task1VAE(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.encoder = Encoder(ld)
    def encode(self, x): mu, _ = self.encoder(x); return mu

In [4]:
class LieAlgebraModule(nn.Module):
    def __init__(self, ld: int):
        super().__init__()
        self.L_lower = nn.Parameter(torch.zeros(ld, ld))
 
    def get_A(self) -> torch.Tensor:
        L = torch.tril(self.L_lower, diagonal=-1)
        return L - L.T   # skew-symmetric
 
    def R(self, theta_rad: float) -> torch.Tensor:
        return torch.linalg.matrix_exp(theta_rad * self.get_A())

In [5]:
@torch.no_grad()
def reynolds_project(
    W:         torch.Tensor,   # (out, in) weight matrix
    A:         torch.Tensor,   # (ld, ld) skew-symmetric Lie algebra generator
    n_group:   int = 12,       # Z_n approximation of SO(2)
    in_dim:    int = None,     # input dimension (may differ from A.shape[0])
    out_dim:   int = None,     # output dimension
) -> torch.Tensor:
    """
    Reynolds operator for SO(2) acting on (out_dim × in_dim) weight matrix.
 
    Formula:  W_proj = (1/n) Σ_{k=0}^{n-1} R_out(θ_k) W R_in(-θ_k)
 
    where θ_k = 2πk/n are the Z_n rotation angles.
 
    For a classifier: only R_in acts on the input latent space (dim=ld).
    The output has no group action (it produces class logits), so R_out = I.
 
    Hence: W_proj = (1/n) Σ_k W · R(-θ_k)   — projects columns onto commutant.
 
    This is mathematically exact for SO(2) as n→∞ and is a valid approximation
    for n=12 (Z_12 ≈ the discrete 30° rotation group we care about).
    """
    ld       = A.shape[0]
    in_d     = W.shape[1]
    out_d    = W.shape[0]
    W_proj   = torch.zeros_like(W)
 
    for k in range(n_group):
        theta_k  = 2 * np.pi * k / n_group
        R_in     = torch.linalg.matrix_exp(-theta_k * A)   # R(-θ_k) for input
 
        # Pad or slice R_in to match W.shape[1] if in_dim != ld
        if in_d == ld:
            R_eff = R_in
        elif in_d > ld:
            # Embed: R acts on first ld dims, identity on rest
            R_eff = torch.eye(in_d, device=A.device)
            R_eff[:ld, :ld] = R_in
        else:
            R_eff = R_in[:in_d, :in_d]   # restrict
 
        W_proj = W_proj + W @ R_eff
 
    return W_proj / n_group
 
 
def commutator_loss(
    W: torch.Tensor,
    A: torch.Tensor
) -> torch.Tensor:
    """
    Soft invariance regulariser for a single weight matrix W and generator A.
 
    The group SO(2) acts ONLY on the latent space R^ld via R(θ) = expm(θA).
    Hidden dimensions carry NO group action.
 
    Layer-specific conditions
    ─────────────────────────
    fc1  W: (hidden, ld)    input is latent → invariance requires W·R(θ) = W
                             Infinitesimally: W @ A = 0  → loss = ||W @ A||²_F
    fc2  W: (hidden, hidden) input is hidden (no group action) → no constraint → 0
    fc3  W: (hidden/2, hidden) same → 0
    fc4  W: (nc, hidden/2)   same → 0
 
    Shape safety: W @ A is valid only when W.shape[1] == A.shape[0] (== ld).
    For all other layers we return 0 — they have no group-theoretic constraint.
    """
    in_d = W.shape[1]
    ld   = A.shape[0]
 
    if in_d == ld:
        # fc1: input has SO(2) action.  Invariance ↔ W @ A = 0.
        WA = W @ A           # (hidden, ld) @ (ld, ld) = (hidden, ld)  ✓
        return (WA * WA).mean()
    else:
        # fc2, fc3, fc4: hidden dims carry no group action → no constraint.
        return torch.zeros(1, device=W.device).squeeze()

In [6]:
class EquivariantLinear(nn.Module):
    """
    Linear layer with optional hard commutant projection after each forward.
    The weight matrix is constrained to commute with R(θ) for all θ.
    """
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)
 
    def project_to_commutant(self, A: torch.Tensor, n_group: int = 12):
        """In-place projection of weight onto commutant of A."""
        with torch.no_grad():
            self.weight.copy_(reynolds_project(self.weight, A, n_group))
 
 
class ACEClassifier(nn.Module):
    """
    Algebra-Commutant Equivariant Classifier.
 
    Architecture: 4-layer MLP identical to baseline, but linear layers
    are instances of EquivariantLinear, allowing commutant projection.
    """
    def __init__(self, ld: int, nc: int, hidden: int = 256):
        super().__init__()
        self.ld = ld
        self.fc1 = EquivariantLinear(ld,     hidden)
        self.fc2 = EquivariantLinear(hidden, hidden)
        self.fc3 = EquivariantLinear(hidden, hidden // 2)
        self.fc4 = EquivariantLinear(hidden // 2, nc)
        self.ln1 = nn.LayerNorm(hidden)
        self.ln2 = nn.LayerNorm(hidden)
 
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.ln1(self.fc1(z)))
        x = F.relu(self.ln2(self.fc2(x)))
        x = F.relu(self.fc3(x))
        return self.fc4(x)
 
    def project_all_layers(self, A: torch.Tensor, n_group: int = 12):
        """Hard projection: project all weight matrices onto commutant."""
        for m in [self.fc1, self.fc2, self.fc3, self.fc4]:
            m.project_to_commutant(A, n_group)
 
    def commutator_loss(self, A: torch.Tensor) -> torch.Tensor:
        """Soft regularisation: sum of commutator losses across all layers."""
        total = torch.zeros(1, device=next(self.parameters()).device).squeeze()
        for m in [self.fc1, self.fc2, self.fc3, self.fc4]:
            total = total + commutator_loss(m.weight, A)
        return total / 4.0
 
    def commutator_norms(self, A: torch.Tensor) -> list:
        """
        Return ||W_l @ A||_F per layer as an invariance diagnostic.
 
        Only fc1 has a ld-dimensional input with SO(2) group action.
        For fc2–fc4 (hidden-dim inputs) the norm is defined as 0.0 since
        there is no group action on the hidden dimensions.
        """
        norms = []
        ld = A.shape[0]
        with torch.no_grad():
            for m in [self.fc1, self.fc2, self.fc3, self.fc4]:
                in_d = m.weight.shape[1]
                if in_d == ld:
                    # fc1: group acts on input → measure ||W @ A||_F
                    wa = m.weight @ A   # (hidden, ld)
                    norms.append(wa.norm().item())
                else:
                    # fc2–fc4: no group action on input → trivially 0
                    norms.append(0.0)
        return norms
 
 
# Standard MLP (baseline for fair comparison)
class MLPClassifier(nn.Module):
    def __init__(self, ld: int, nc: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ld, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, nc),
        )
    def forward(self, z): return self.net(z)

In [7]:
def load_vae(path, ld):
    vae = Task1VAE(ld).to(device)
    if os.path.exists(path):
        ck  = torch.load(path, map_location=device, weights_only=False)
        # Load only encoder weights
        full_sd = ck.get("model", ck)
        enc_sd  = {k.replace("encoder.", ""): v
                   for k, v in full_sd.items() if k.startswith("encoder.")}
        if enc_sd:
            vae.encoder.load_state_dict(enc_sd)
        else:
            # Try loading full VAE and keep encoder
            from collections import OrderedDict
            vae_tmp = type('_VAE', (), {'encoder': vae.encoder})()
            try:
                vae.encoder.load_state_dict(
                    {k.replace("encoder.", ""): v
                     for k, v in full_sd.items() if "encoder" in k}, strict=False)
            except Exception:
                pass
        print(f"  ✓ VAE encoder loaded (epoch {ck.get('epoch','?')})")
    else:
        print(f"  ⚠  VAE ckpt not found: {path}")
    vae.eval()
    for p in vae.parameters(): p.requires_grad = False
    return vae
 
def load_lie_algebra(path, ld):
    """
    Extract the learned skew-symmetric generator A from Task-2 checkpoint.
 
    Task-2 (e2e-task2-v4) saves:
        torch.save({"T": T_net.state_dict(), "psi": psi.state_dict(),
                    "epoch": ep, "val_score": best_val}, ...)
    So the HybridLieTransform state dict lives under key "T".
    Its Lie algebra parameters are under keys "lie.L_lower" inside that dict.
    """
    lie = LieAlgebraModule(ld).to(device)
    if not os.path.exists(path):
        print(f"  ⚠  Task-2 ckpt not found: {path} – using random Lie algebra")
        lie.eval()
        for p in lie.parameters(): p.requires_grad = False
        return lie
 
    ck = torch.load(path, map_location=device, weights_only=False)
    print(f"  Task-2 ckpt keys: {list(ck.keys())}")
 
    # Task-2 saves T_net under key "T"
    # Try each possible top-level key in priority order
    t_net_sd = None
    for top_key in ("T", "T_net", "model"):
        if top_key in ck:
            t_net_sd = ck[top_key]
            print(f"  Found T_net state dict under key '{top_key}'")
            break
 
    if t_net_sd is None:
        # Flat checkpoint — the whole thing is a state dict
        t_net_sd = ck
        print(f"  Treating checkpoint as flat state dict")
 
    # Extract only lie.* keys → strip "lie." prefix for LieAlgebraModule
    lie_sd = {k[len("lie."):]: v
              for k, v in t_net_sd.items() if k.startswith("lie.")}
 
    if lie_sd:
        missing, unexpected = lie.load_state_dict(lie_sd, strict=False)
        A_check = lie.get_A()
        frob    = A_check.norm().item()
        skew_err = (A_check + A_check.T).norm().item()
        print(f"  ✓ Lie algebra A loaded  ||A||_F={frob:.4f}  "
              f"skew-sym error={skew_err:.2e}")
        if missing:
            print(f"    missing keys: {missing}")
    else:
        print(f"  ⚠  No 'lie.*' keys found in checkpoint.  "
              f"Available keys (first 10): {list(t_net_sd.keys())[:10]}")
        print(f"  ⚠  Using zero Lie algebra — commutant constraint will be trivial")
 
    lie.eval()
    for p in lie.parameters(): p.requires_grad = False
    return lie
 
def load_h5(path):
    print(f"  loading {os.path.basename(path)} ...", end=" ", flush=True)
    t0 = time.time()
    with h5py.File(path, "r") as f:
        imgs   = torch.from_numpy(f["images"][:].astype("float32")).unsqueeze(1).clamp(0., 1.)
        labels = torch.from_numpy(f["labels"][:].astype("int64"))
        angles = torch.from_numpy(f["angles"][:].astype("int64"))
    print(f"done ({time.time()-t0:.1f}s)")
    return imgs, labels, angles

In [8]:
def train_acec(
    Z_train:    torch.Tensor,
    L_train:    torch.Tensor,
    A_frozen:   torch.Tensor,           # (ld, ld) Lie generator (frozen)
    latent_dim: int,
    n_classes:  int,
    epochs:     int,
    batch_size: int,
    lr:         float,
    lambda_comm: float,
    projection_mode: str,
    n_reynolds: int,
    tag: str = "",
    proj_every: int = 5,                # hard-project weights every N batches
) -> tuple:
    """
    Train ACEC with specified projection mode.
 
    proj_every: for hard/both modes, how many gradient steps between projections.
                Projecting every single step (proj_every=1) destroys gradient
                signal too aggressively and causes accuracy collapse.
                proj_every=5 gives the optimiser room to explore before constraining.
 
    Returns (model, history dict including per-layer commutator norms).
    """
    clf    = ACEClassifier(latent_dim, n_classes).to(device)
    opt    = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(Z_train, L_train),
                        batch_size=batch_size, shuffle=True)
 
    history = {
        "train_acc": [], "train_loss": [], "comm_loss": [],
        "comm_norms_l1": [], "comm_norms_l2": [],
        "comm_norms_l3": [], "comm_norms_l4": []
    }
    print(f"\n  Training {tag} for {epochs} epochs (mode={projection_mode}) …")
    global_step = 0
 
    for ep in range(1, epochs + 1):
        clf.train()
        correct = total = loss_sum = comm_sum = 0
 
        for z, lbl in loader:
            z, lbl = z.to(device), lbl.to(device)
            opt.zero_grad()
            logits = clf(z)
            ce     = F.cross_entropy(logits, lbl)
 
            # Soft commutator regularisation
            if projection_mode in ("soft", "both"):
                l_comm = clf.commutator_loss(A_frozen)
                loss   = ce + lambda_comm * l_comm
                comm_sum += l_comm.item() * z.size(0)
            else:
                loss     = ce
                l_comm   = torch.zeros(1)
 
            loss.backward()
            nn.utils.clip_grad_norm_(clf.parameters(), 5.0)
            opt.step()
            global_step += 1
 
            # Hard projection: project every proj_every steps to avoid collapse
            if projection_mode in ("hard", "both") and global_step % proj_every == 0:
                clf.project_all_layers(A_frozen, n_reynolds)
 
            correct  += (logits.detach().argmax(1) == lbl).sum().item()
            total    += z.size(0)
            loss_sum += ce.item() * z.size(0)
 
        sched.step()
        clf.eval()
        norms = clf.commutator_norms(A_frozen)
        history["train_acc"].append(correct / total)
        history["train_loss"].append(loss_sum / total)
        history["comm_loss"].append(comm_sum / max(total, 1))
        for i, key in enumerate(["comm_norms_l1", "comm_norms_l2",
                                  "comm_norms_l3", "comm_norms_l4"]):
            history[key].append(norms[i])
 
        if ep % 10 == 0 or ep == 1:
            print(f"    ep {ep:3d}/{epochs}  acc={correct/total:.4f}  "
                  f"loss={loss_sum/total:.4f}  "
                  f"[W1,A]={norms[0]:.3f}  [W4,A]={norms[3]:.3f}")
 
    clf.eval()
    return clf, history
 
 
def train_baseline(Z_train, L_train, ld, nc, epochs, batch_size, lr):
    """Train standard MLP baseline for comparison."""
    clf    = MLPClassifier(ld, nc).to(device)
    opt    = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(Z_train, L_train),
                        batch_size=batch_size, shuffle=True)
    hist   = {"train_acc": [], "train_loss": []}
    print(f"\n  Training MLP-baseline for {epochs} epochs …")
    for ep in range(1, epochs + 1):
        clf.train()
        correct = total = loss_sum = 0
        for z, lbl in loader:
            z, lbl = z.to(device), lbl.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(clf(z), lbl)
            loss.backward(); opt.step()
            correct  += (clf(z).detach().argmax(1) == lbl).sum().item()
            total    += z.size(0)
            loss_sum += loss.item() * z.size(0)
        sched.step()
        hist["train_acc"].append(correct / total)
        hist["train_loss"].append(loss_sum / total)
        if ep % 10 == 0 or ep == 1:
            print(f"    ep {ep:3d}/{epochs}  acc={correct/total:.4f}")
    clf.eval()
    return clf, hist

In [9]:
@torch.no_grad()
def per_angle_accuracy_latent(
    clf, Z_by_angle: dict, labels: torch.Tensor, batch: int = 512
) -> dict:
    results = {}
    for angle, Z in Z_by_angle.items():
        correct = total = 0
        for i in range(0, len(Z), batch):
            z_b  = Z[i:i+batch].to(device)
            l_b  = labels[i:i+batch].to(device)
            pred = clf(z_b).argmax(1)
            correct += (pred == l_b).sum().item()
            total   += z_b.size(0)
        results[angle] = correct / total
    return results
 
 
@torch.no_grad()
def commutant_decomposition(
    W: torch.Tensor,
    A: torch.Tensor,
    n_group: int = 12
) -> tuple:
    """
    Decompose W = W_comm + W_anti
    W_comm = Reynolds(W)  — invariant part (commutes with all R(θ))
    W_anti = W - W_comm   — equivariant part
 
    The Reynolds operator is only meaningful when W.shape[1] == A.shape[0]
    (i.e., the input dimension of the layer equals the latent/group dimension).
    For other layers, W_comm = W and ratio = 1.0 by convention.
 
    Returns (||W_comm||_F, ||W_anti||_F, ratio = ||W_comm||/||W||)
    """
    in_d = W.shape[1]
    ld   = A.shape[0]
 
    if in_d != ld:
        # No group action on this layer's input → fully in commutant by convention
        norm_w = W.norm().item()
        return norm_w, 0.0, 1.0
 
    W_comm = reynolds_project(W, A, n_group)
    W_anti = W - W_comm
    norm_w    = W.norm().item()
    norm_comm = W_comm.norm().item()
    norm_anti = W_anti.norm().item()
    ratio = norm_comm / max(norm_w, 1e-8)
    return norm_comm, norm_anti, ratio

In [10]:
DARK, PANEL = "#0d0d0d", "#1a1a2e"
CMAP = ["#e94560", "#0f9b8e", "#f5a623", "#9b59b6", "#2ecc71", "#3498db"]
 
def _ax(ax, title="", xl="", yl=""):
    ax.set_facecolor(PANEL); ax.tick_params(colors="white")
    for s in ax.spines.values(): s.set_edgecolor("#444")
    if title: ax.set_title(title, color="white", fontsize=9)
    if xl:    ax.set_xlabel(xl, color="white", fontsize=8)
    if yl:    ax.set_ylabel(yl, color="white", fontsize=8)
 
 
def plot_equivariant_comparison(
    angles, results_dict, history_dict, comm_ratios, out_path
):
    """
    results_dict: {method_name: {angle: acc}}
    history_dict: {method_name: history}
    comm_ratios:  {method_name: [l1, l2, l3, l4]}  — invariant weight ratio per layer
    """
    fig = plt.figure(figsize=(22, 14), facecolor=DARK)
    fig.suptitle("Exp 4 — Algebra-Commutant Equivariant Classifier (ACEC)\n"
                 "Novel: Lie-algebra commutant constraint enforced on each weight matrix",
                 color="white", fontsize=12, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)
    x  = np.array(angles)
 
    # Panel 1: per-angle accuracy comparison
    ax1 = fig.add_subplot(gs[0, :2])
    _ax(ax1, "Per-Angle Accuracy: Baseline vs ACEC Variants",
        "Rotation Angle θ (°)", "Accuracy")
    for i, (name, acc) in enumerate(results_dict.items()):
        y  = np.array([acc[a] for a in angles])
        ls = "--" if "baseline" in name.lower() else "-"
        ax1.plot(x, y, ls, color=CMAP[i % len(CMAP)], lw=2.2,
                 label=f"{name}  [{y.mean():.4f}]")
    ax1.set_xticks(x); ax1.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax1.legend(facecolor=PANEL, labelcolor="white", fontsize=8)
    ax1.set_ylim(0, 1.05)
 
    # Panel 2: mean accuracy bar
    ax2 = fig.add_subplot(gs[0, 2])
    _ax(ax2, "Mean Accuracy Summary", "", "Mean Accuracy (all θ)")
    names = list(results_dict.keys())
    means = [np.mean(list(results_dict[n].values())) for n in names]
    bars  = ax2.bar(range(len(names)), means, color=CMAP[:len(names)], alpha=0.85, edgecolor=DARK)
    ax2.set_xticks(range(len(names)))
    ax2.set_xticklabels(names, color="white", fontsize=8, rotation=20, ha="right")
    for bar, v in zip(bars, means):
        ax2.text(bar.get_x()+bar.get_width()/2, v+0.005,
                 f"{v:.4f}", ha="center", color="white", fontsize=8, fontweight="bold")
    ax2.set_ylim(0, 1.15)
 
    # Panel 3: commutator norms during training
    ax3 = fig.add_subplot(gs[1, 0])
    _ax(ax3, "Commutator Norms ||[W_l, A]||_F During Training\n(converging to 0 = algebra constraint satisfied)",
        "Epoch", "||[W, A]||_F")
    for name, hist in history_dict.items():
        if "comm_norms_l1" in hist:
            for li, (lname, col) in enumerate([("L1", CMAP[0]), ("L2", CMAP[1]),
                                               ("L3", CMAP[2]), ("L4", CMAP[3])]):
                key = f"comm_norms_l{li+1}"
                ax3.plot(hist[key], color=col, lw=1.5, alpha=0.8,
                         label=f"{name}/{lname}")
    ax3.legend(facecolor=PANEL, labelcolor="white", fontsize=6, ncol=2)
 
    # Panel 4: commutant weight ratio per layer
    ax4 = fig.add_subplot(gs[1, 1])
    _ax(ax4, "Commutant Weight Ratio per Layer\n||W_commutant||/||W||  (higher = more invariant)",
        "Layer", "Ratio")
    layer_labels = ["L1", "L2", "L3", "L4"]
    for i, (name, ratios) in enumerate(comm_ratios.items()):
        ax4.plot(range(4), ratios, "o-", color=CMAP[i], lw=2, label=name)
    ax4.axhline(1.0, color="white", ls=":", lw=1, alpha=0.4, label="perfect invariance")
    ax4.set_xticks(range(4)); ax4.set_xticklabels(layer_labels, color="white", fontsize=9)
    ax4.legend(facecolor=PANEL, labelcolor="white", fontsize=8)
    ax4.set_ylim(0, 1.15)
 
    # Panel 5: training accuracy curves
    ax5 = fig.add_subplot(gs[1, 2])
    _ax(ax5, "Training Accuracy Curves", "Epoch", "Train Accuracy")
    for i, (name, hist) in enumerate(history_dict.items()):
        ax5.plot(hist["train_acc"], color=CMAP[i], lw=2, label=name)
    ax5.legend(facecolor=PANEL, labelcolor="white", fontsize=8)
    ax5.set_ylim(0, 1.05)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")
 
 
def plot_commutator_analysis(
    A_np: np.ndarray,
    models_dict: dict,
    out_path: str
):
    """Visualise the Lie algebra A and W-decompositions for each trained ACEC model."""
    acec_count = sum(1 for m in models_dict.values() if isinstance(m, ACEClassifier))
    n_cols     = max(acec_count, 1)   # at least 1 column for the Lie algebra panel
    fig = plt.figure(figsize=(5 + 4 * n_cols, 10), facecolor=DARK)
    fig.suptitle("Algebra-Commutant Analysis\n"
                 "Top: Lie generator A  |  Bottom: W₁ decomposition into commutant + anti-commutant",
                 color="white", fontsize=11, fontweight="bold")
    gs = gridspec.GridSpec(2, n_cols + 1, figure=fig, hspace=0.4, wspace=0.3)
 
    # Panel: Lie algebra A
    ax0 = fig.add_subplot(gs[:, 0])
    _ax(ax0, "Lie Algebra A\n(skew-symmetric generator)", "j", "i")
    vmax = max(abs(A_np).max(), 1e-6)
    im   = ax0.imshow(A_np, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax0, fraction=0.046)
    ax0.set_title("Lie Algebra A", color="white", fontsize=9)
 
    # Per model: layer 1 weight decomposition.
    # Only ACEClassifier has .fc1; MLPClassifier uses .net — skip non-ACEC models.
    ld  = A_np.shape[0]
    A_t = torch.tensor(A_np).to(device)
    acec_models = [(n, m) for n, m in models_dict.items()
                   if isinstance(m, ACEClassifier)]
    for col, (name, model) in enumerate(acec_models):
        ax_comm = fig.add_subplot(gs[0, col + 1])
        ax_anti = fig.add_subplot(gs[1, col + 1])
        _ax(ax_comm, f"{name}\nW₁ commutant part", "j", "i")
        _ax(ax_anti, "W₁ anti-commutant part", "j", "i")
        W1  = model.fc1.weight.detach()          # (hidden, ld)
        W_c = reynolds_project(W1, A_t).cpu().numpy()
        W_a = W1.cpu().numpy() - W_c
        # Visualise first ld×ld block
        vm  = max(abs(W_c[:ld, :ld]).max(), 1e-6)
        im1 = ax_comm.imshow(W_c[:ld, :ld], cmap="RdBu_r", vmin=-vm, vmax=vm)
        plt.colorbar(im1, ax=ax_comm, fraction=0.046)
        vm2 = max(abs(W_a[:ld, :ld]).max(), 1e-6)
        im2 = ax_anti.imshow(W_a[:ld, :ld], cmap="RdBu_r", vmin=-vm2, vmax=vm2)
        plt.colorbar(im2, ax=ax_anti, fraction=0.046)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")

In [11]:
def main():
    args = get_args()
    os.makedirs(args.out_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  Bonus Task · Exp 4: Algebra-Commutant Equivariant Classifier")
    print(f"  device = {device}")
    print(f"  projection_mode = {args.projection_mode}")
    print(f"  output → {args.out_dir}")
    print(f"{'='*60}\n")
 
    # ── Load models ───────────────────────────────────────────────────────────
    vae = load_vae(args.task1_ckpt, args.latent_dim)
    lie = load_lie_algebra(args.task2_ckpt, args.latent_dim)
    A_frozen = lie.get_A().detach()
    print(f"  Lie algebra A: ||A||_F = {A_frozen.norm().item():.4f}  "
          f"  skew-sym error ||A+A^T||_F = {(A_frozen + A_frozen.T).norm().item():.2e}")
 
    # ── Load data ─────────────────────────────────────────────────────────────
    train_imgs, train_lbls, train_angs = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_train.h5"))
    test_imgs,  test_lbls,  test_angs  = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_test.h5"))
    ANGLES  = sorted(test_angs.unique().tolist())
    DIGITS  = sorted(test_lbls.unique().tolist())
    dig2cls = {d: i for i, d in enumerate(DIGITS)}
 
    # Pre-encode training (0° only — no augmentation, like exp1 NoAug)
    print("\n[1] Pre-encoding training latents (0° only) …")
    mask_0   = train_angs == 0
    tr_imgs0 = train_imgs[mask_0]
    tr_lbls0 = train_lbls[mask_0]
    tr_cls0  = torch.tensor([dig2cls[l.item()] for l in tr_lbls0])
    Z_train  = []
    for i in tqdm(range(0, len(tr_imgs0), 512), desc="  encoding train", leave=False):
        Z_train.append(vae.encode(tr_imgs0[i:i+512].to(device)).cpu())
    Z_train = torch.cat(Z_train)
 
    # Pre-encode test (0° samples, then rotate for evaluation)
    print("[2] Pre-encoding test latents per angle …")
    mask_0_te = test_angs == 0
    te_imgs0  = test_imgs[mask_0_te]
    te_lbls0  = test_lbls[mask_0_te]
    te_cls0   = torch.tensor([dig2cls[l.item()] for l in te_lbls0])
 
    Z_by_angle = {}
    for angle in tqdm(ANGLES, desc="  encoding test angles", leave=False):
        chunks = []
        for i in range(0, len(te_imgs0), 512):
            chunk = te_imgs0[i:i+512]
            if angle != 0:
                chunk = tvrotate(chunk, float(angle),
                                 interpolation=transforms.InterpolationMode.BILINEAR, fill=[0.])
            chunks.append(vae.encode(chunk.to(device)).cpu())
        Z_by_angle[angle] = torch.cat(chunks)
 
    # ── Train classifiers ─────────────────────────────────────────────────────
    print("\n[3] Training classifiers …")
 
    # Baseline MLP (no commutant constraint)
    clf_base, hist_base = train_baseline(
        Z_train, tr_cls0, args.latent_dim, args.n_classes,
        args.epochs, args.batch_size, args.lr)
 
    # ACEC variants
    acec_variants = {}
    if args.projection_mode == "both":
        modes_to_test = ["soft", "hard", "both"]
    else:
        modes_to_test = [args.projection_mode]
 
    # Always test all 3 ACEC variants for comparison
    for mode in ["soft", "hard", "both"]:
        clf_v, hist_v = train_acec(
            Z_train, tr_cls0, A_frozen,
            args.latent_dim, args.n_classes,
            args.epochs, args.batch_size, args.lr,
            args.lambda_commutator, mode, args.n_reynolds,
            tag=f"ACEC-{mode}",
            proj_every=5)   # project every 5 steps to avoid gradient collapse
        acec_variants[mode] = (clf_v, hist_v)
 
    # ── Evaluate ──────────────────────────────────────────────────────────────
    print("\n[4] Evaluating per-angle accuracy …")
    results   = {}
    histories = {}
 
    acc_base = per_angle_accuracy_latent(clf_base, Z_by_angle, te_cls0)
    results["Baseline-MLP"] = acc_base
    histories["Baseline-MLP"] = hist_base
 
    for mode, (clf_v, hist_v) in acec_variants.items():
        acc_v = per_angle_accuracy_latent(clf_v, Z_by_angle, te_cls0)
        results[f"ACEC-{mode}"] = acc_v
        histories[f"ACEC-{mode}"] = hist_v
 
    # ── Commutant analysis ────────────────────────────────────────────────────
    print("\n[5] Computing commutant decompositions …")
    A_frozen_cpu = A_frozen.cpu()
    comm_ratios  = {}
 
    # Baseline
    ratios_base = []
    for fc in [clf_base.net[0], clf_base.net[3], clf_base.net[6], clf_base.net[8]]:
        _, _, r = commutant_decomposition(fc.weight.detach().cpu(), A_frozen_cpu)
        ratios_base.append(r)
    comm_ratios["Baseline-MLP"] = ratios_base
 
    for mode, (clf_v, _) in acec_variants.items():
        ratios = []
        for fc in [clf_v.fc1, clf_v.fc2, clf_v.fc3, clf_v.fc4]:
            _, _, r = commutant_decomposition(fc.weight.detach().cpu(), A_frozen_cpu)
            ratios.append(r)
        comm_ratios[f"ACEC-{mode}"] = ratios
 
    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  RESULTS SUMMARY — Exp 4")
    print(f"{'='*60}")
    for name, acc in results.items():
        m = np.mean(list(acc.values()))
        rat = np.mean(comm_ratios.get(name, [0])*4 if not isinstance(comm_ratios.get(name,[]), list)
                      else comm_ratios.get(name, [0, 0, 0, 0]))
        print(f"  {name:<20}  mean acc = {m:.4f}  "
              f"||W_comm||/||W|| = {rat:.3f}")
 
    best_acec = max(
        [(name, np.mean(list(acc.values()))) for name, acc in results.items()
         if "ACEC" in name],
        key=lambda x: x[1])
    base_mean = np.mean(list(results["Baseline-MLP"].values()))
    print(f"\n  Best ACEC variant: {best_acec[0]}  acc = {best_acec[1]:.4f}  "
          f"Δ = {best_acec[1] - base_mean:+.4f}")
 
    # ── Save artefacts ────────────────────────────────────────────────────────
    print("\n[6] Saving results and figures …")
    save_dict = {"angles": ANGLES}
    for name, acc in results.items():
        save_dict[f"acc_{name.replace(' ','_')}"] = [acc[a] for a in ANGLES]
    for name, ratios in comm_ratios.items():
        save_dict[f"comm_ratio_{name.replace(' ','_')}"] = ratios
    np.savez(os.path.join(args.out_dir, "acec_results.npz"), **save_dict)
 
    # Save models
    save_states = {"baseline": clf_base.state_dict()}
    for mode, (clf_v, _) in acec_variants.items():
        save_states[f"acec_{mode}"] = clf_v.state_dict()
    torch.save(save_states, os.path.join(args.out_dir, "classifiers.pt"))
 
    plot_equivariant_comparison(
        ANGLES, results, histories, comm_ratios,
        os.path.join(args.out_dir, "fig_equivariant_comparison.png"))
 
    A_np = A_frozen.cpu().numpy()
    models_for_plot = {"Baseline": clf_base}
    for mode, (clf_v, _) in acec_variants.items():
        models_for_plot[f"ACEC-{mode}"] = clf_v
    plot_commutator_analysis(
        A_np, models_for_plot,
        os.path.join(args.out_dir, "fig_commutator_analysis.png"))
 
    print(f"\n✅  Exp 4 complete.  Outputs → {args.out_dir}/")
    print(f"    Best ACEC = {best_acec[0]}  mean accuracy = {best_acec[1]:.4f}")
    print(f"    Improvement over no-invariance baseline = {best_acec[1]-base_mean:+.4f}")

In [12]:
if __name__ == "__main__":
    main()


  Bonus Task · Exp 4: Algebra-Commutant Equivariant Classifier
  device = cuda
  projection_mode = both
  output → ./bonus_outputs/exp4

  ✓ VAE encoder loaded (epoch 47)
  Task-2 ckpt keys: ['T', 'psi', 'epoch', 'val_score']
  Found T_net state dict under key 'T'
  ✓ Lie algebra A loaded  ||A||_F=0.4466  skew-sym error=0.00e+00
  Lie algebra A: ||A||_F = 0.4466    skew-sym error ||A+A^T||_F = 0.00e+00
  loading rotated_mnist_train.h5 ... done (2.6s)
  loading rotated_mnist_test.h5 ... done (0.4s)

[1] Pre-encoding training latents (0° only) …


  encoding train:   0%|          | 0/25 [00:00<?, ?it/s]

[2] Pre-encoding test latents per angle …


  encoding test angles:   0%|          | 0/12 [00:00<?, ?it/s]


[3] Training classifiers …

  Training MLP-baseline for 40 epochs …
    ep   1/40  acc=0.9817
    ep  10/40  acc=0.9920
    ep  20/40  acc=0.9927
    ep  30/40  acc=0.9931
    ep  40/40  acc=0.9932

  Training ACEC-soft for 40 epochs (mode=soft) …
    ep   1/40  acc=0.9911  loss=0.0844  [W1,A]=1.002  [W4,A]=0.000
    ep  10/40  acc=0.9920  loss=0.0274  [W1,A]=0.977  [W4,A]=0.000
    ep  20/40  acc=0.9928  loss=0.0240  [W1,A]=0.966  [W4,A]=0.000
    ep  30/40  acc=0.9934  loss=0.0192  [W1,A]=0.965  [W4,A]=0.000
    ep  40/40  acc=0.9939  loss=0.0177  [W1,A]=0.965  [W4,A]=0.000

  Training ACEC-hard for 40 epochs (mode=hard) …
    ep   1/40  acc=0.8765  loss=0.3086  [W1,A]=0.462  [W4,A]=0.000
    ep  10/40  acc=0.9902  loss=0.0393  [W1,A]=0.140  [W4,A]=0.000
    ep  20/40  acc=0.9908  loss=0.0385  [W1,A]=0.104  [W4,A]=0.000
    ep  30/40  acc=0.9894  loss=0.0438  [W1,A]=0.085  [W4,A]=0.000
    ep  40/40  acc=0.9539  loss=0.1478  [W1,A]=0.074  [W4,A]=0.000

  Training ACEC-both for 40 ep